In [ ]:
import pandas as pd
import torch
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from torch import nn, optim

In [10]:
df = pd.read_csv('../dataset/extended_amazon_products.csv')
df.head(4)


,text,category
0,Apple Watch Series 9 GPS smartwatch with fitne...,Electronics_SmartWatch
1,Samsung Galaxy Watch 6 AMOLED smartwatch,Electronics_SmartWatch
2,Noise ColorFit smart watch heart rate monitor,Electronics_SmartWatch
3,Fire-Boltt smart watch bluetooth calling,Electronics_SmartWatch


In [11]:
df.tail(4)

,text,category
176,Pet vitamins supplements,Pet_Supplies
177,Cat scratching post,Pet_Supplies
178,Pet toy squeaky,Pet_Supplies
179,Dog harness comfort fit,Pet_Supplies


In [12]:
x = df["text"].values
y = df["category"].values

In [13]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Text → numbers
vectorizer = TfidfVectorizer(max_features=5000)
x_vectorized = vectorizer.fit_transform(x).toarray()

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    x_vectorized, y_encoded, test_size=0.2, random_state=42
)

In [ ]:
# Train model
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

In [15]:
# Convert to tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
import sys
sys.path.append("..")

from text_model import TextClassifier
import torch.optim as optim
import torch

model = TextClassifier(
    input_dim=x_train.shape[1],
    num_classes=len(label_encoder.classes_)
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()

# Training loop
for epoch in range(10):
    optimizer.zero_grad()
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Save everything
torch.save({
    "model_state": model.state_dict(),
    "vectorizer": vectorizer,
    "label_encoder": label_encoder
}, "artifacts.pth")
#add/docs while running the server

Epoch 1, Loss: 2.1987
Epoch 2, Loss: 2.1903
Epoch 3, Loss: 2.1821
Epoch 4, Loss: 2.1738
Epoch 5, Loss: 2.1655
Epoch 6, Loss: 2.1571
Epoch 7, Loss: 2.1484
Epoch 8, Loss: 2.1394
Epoch 9, Loss: 2.1300
Epoch 10, Loss: 2.1203
